# XIOSYNC Worker

This runtime is managed by the **XIOSYNC control plane**.  
All configuration is delivered securely — nothing to edit here.

| What | How |
|---|---|
| Chrome profiles | Google Drive FUSE (XIODriveFS, org-zero-drive) |
| Cookie fallback | XIOSYNC vault (Org Zero) |
| Mesh networking | Tailscale |
| Session control | XIOSYNC → xiorun_agent `:9300` over Tailscale |
| Exit node (residential IP) | PPPoE via XIOSYNC xiogrid |

---

## Starting a worker

1. Get a bootstrap URL from the XIOSYNC admin panel  
   (`POST /api/v1/workers/bootstrap-tokens`)
2. Open the generated `colab_url` — it always includes `forceEdit=true&sandboxMode=true`  
   so the notebook runs in **playground mode** (no auto-save, isolated from the original file)
3. Run All (`Runtime → Run all`)
4. You may close the browser after Cell 3 starts

**Direct open URL format:**
```
https://colab.research.google.com/drive/1Kk_VSYGuNWeVqAhycl-c_lTQRNLhT9Op
  ?xiosync_base=https://karmas-mac-mini.taildd8b9a.ts.net
  &bootstrap_token=<TOKEN>
  #forceEdit=true&sandboxMode=true
```

> ⚠️ **Do not share your bootstrap token.** It authenticates a worker to your XIOSYNC org.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 0.5 — Notebook self-update (run-only, do not edit)
# Checks XIOSYNC for a newer version of this notebook and patches
# the Drive copy in-place so next boot is always on the latest version.
# Requires: Google Drive already mounted (run after drive auth if needed).
# Safe to skip — boot continues even if update check fails.
# ══════════════════════════════════════════════════════════════════════════════
import os as _os, json as _json, urllib.request as _req, hashlib as _hs

_NB_NAME     = "xiosync-worker.ipynb"
_DRIVE_NB    = f"/content/drive/MyDrive/XIOSYNC-Shared/{_NB_NAME}"
_XIOSYNC_BASE = ""

# Read XIOSYNC_BASE from URL params if injected (same logic as Cell 1)
try:
    import IPython as _ipy, json as _jj, time as _tt
    _ipy.display.display(_ipy.display.Javascript("""
        const p = new URLSearchParams(window.location.search);
        const b = p.get('xiosync_base') || '';
        const xhr = new XMLHttpRequest();
        xhr.open('POST', '/nbextensions/xiosync_base_tmp', false);
        xhr.send(JSON.stringify({b}));
    """))
    _tt.sleep(0.5)
except Exception:
    pass

_XIOSYNC_BASE = _XIOSYNC_BASE or _os.environ.get("XIOSYNC_BASE", "")

# Try env var written by a previous cell run or system
if not _XIOSYNC_BASE:
    try:
        _XIOSYNC_BASE = open("/tmp/xio_xiosync_base").read().strip()
    except Exception:
        pass

def _nb_self_update(base: str) -> None:
    if not base:
        print("⏭️  Notebook update check skipped (XIOSYNC_BASE unknown at this stage)")
        return
    try:
        _hash_url = f"{base}/api/v1/workers/notebook-hash"
        _remote   = _json.loads(_req.urlopen(_hash_url, timeout=10).read())
        _remote_sha = _remote.get("sha256", "")

        _local_sha = ""
        if _os.path.exists(_DRIVE_NB):
            with open(_DRIVE_NB, "rb") as _f:
                _local_sha = _hs.sha256(_f.read()).hexdigest()

        if _remote_sha and _remote_sha != _local_sha:
            print(f"📦 Notebook update available → fetching…", end=" ")
            _nb_url  = f"{base}/api/v1/workers/notebook.ipynb"
            _nb_data = _req.urlopen(_nb_url, timeout=30).read()
            # Write to Drive FUSE (auto-syncs to Google Drive)
            _os.makedirs(_os.path.dirname(_DRIVE_NB), exist_ok=True)
            with open(_DRIVE_NB, "wb") as _out:
                _out.write(_nb_data)
            print(f"OK (sha256={_remote_sha[:12]}…)")
            print("ℹ️  Notebook updated in Drive. New sessions will use the latest version.")
        else:
            print(f"✅ Notebook up-to-date (sha256={_local_sha[:12]}…)")
    except Exception as _e:
        print(f"⚠️  Notebook update check failed ({_e}) — continuing")

_nb_self_update(_XIOSYNC_BASE)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Bootloader (run-only, do not edit)
# Reads the bootstrap token from the URL query parameters, fetches the full
# worker config from XIOSYNC, then starts the xiorun_agent.
# ══════════════════════════════════════════════════════════════════════════════
import json, os, sys, urllib.request, urllib.parse

# ── Read bootstrap params from URL (injected by Colab when opening the link) ─
# When opened via the admin-generated URL:
#   https://colab.research.google.com/drive/{FILE_ID}
#     ?xiosync_base=http://100.x.x.x:8001&bootstrap_token=xxx
#     #forceEdit=true&sandboxMode=true
# Colab exposes URL params via colab.output or via notebook metadata.
# We read them from the URL using javascript injection.

XIOSYNC_BASE    = ''
BOOTSTRAP_TOKEN = ''

try:
    from google.colab import output as _output  # noqa
    import IPython as _ipy

    # Inject JS to read URL query params and write them to a tmp file
    _ipy.display.display(_ipy.display.Javascript("""
        const params = new URLSearchParams(window.location.search);
        const base   = params.get('xiosync_base') || '';
        const token  = params.get('bootstrap_token') || '';
        const payload = JSON.stringify({xiosync_base: base, bootstrap_token: token});
        const req = new XMLHttpRequest();
        req.open('POST', '/nbextensions/xiosync_params', false);
        req.send(payload);
    """))
    import time as _time
    _time.sleep(1)  # Let JS execute
except Exception:
    pass

# Fallback: read from environment (set by the URL-launched Colab runtime)
XIOSYNC_BASE    = XIOSYNC_BASE    or os.environ.get('XIOSYNC_BASE', '')
BOOTSTRAP_TOKEN = BOOTSTRAP_TOKEN or os.environ.get('XIOSYNC_BOOTSTRAP_TOKEN', '')

# Last fallback: prompt user (happens if opened directly without the token URL)
if not BOOTSTRAP_TOKEN:
    try:
        _raw = input(
            '\n'
            '⚠️  No bootstrap token detected.\n'
            'Paste the bootstrap token from XIOSYNC admin panel, then press Enter.\n'
            'Format: xiosync_base=http://...|token=<token>\n'
            '> '
        ).strip()
        if '|' in _raw:
            _parts = dict(p.split('=', 1) for p in _raw.split('|'))
            XIOSYNC_BASE    = _parts.get('xiosync_base', XIOSYNC_BASE)
            BOOTSTRAP_TOKEN = _parts.get('token', '')
        else:
            BOOTSTRAP_TOKEN = _raw  # treat as bare token if base already known
    except Exception:
        pass

if not BOOTSTRAP_TOKEN or not XIOSYNC_BASE:
    print('❌ Missing bootstrap token or XIOSYNC base URL.')
    print('   Get a bootstrap URL from the XIOSYNC admin panel and open it directly.')
    raise SystemExit(1)

# ── Fetch boot.py from XIOSYNC repo ─────────────────────────────────────────
_BOOT_URL = f'{XIOSYNC_BASE}/api/v1/workers/boot.py'
print(f'Fetching boot.py…', end=' ')
_boot_fetched = False
_boot_urls = [_BOOT_URL]  # primary; fallback URLs appended below
for _attempt, _url in enumerate(_boot_urls + [_BOOT_URL], 1):
    try:
        _boot_src = urllib.request.urlopen(_url, timeout=90).read().decode()
        with open('/tmp/xio_boot.py', 'w') as _f:
            _f.write(_boot_src)
        print(f'OK (attempt {_attempt})')
        _boot_fetched = True
        break
    except Exception as _e:
        print(f'WARN attempt {_attempt} ({type(_e).__name__}) — retrying…')
if not _boot_fetched:
    _local = '/content/xiosync-worker/colab/boot.py'
    if os.path.exists(_local):
        import shutil as _sh
        _sh.copy(_local, '/tmp/xio_boot.py')
        print('OK (local copy)')
    else:
        raise RuntimeError(f'Cannot fetch boot.py after retries.')

# ── Execute boot.py — it fetches full config via bootstrap token ─────────────
with open('/tmp/xio_boot.py') as _f:
    exec(
        compile(_f.read(), 'boot.py', 'exec'),
        {
            'BOOTSTRAP_TOKEN': BOOTSTRAP_TOKEN,
            'XIOSYNC_BASE':    XIOSYNC_BASE,
            '__name__':        '__boot__',
        }
    )

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Keep-Alive (run-only, do not edit)
# xiorun_agent, heartbeat, and watchdog threads are running as daemons.
# You can safely close the browser tab after this cell starts.
# ══════════════════════════════════════════════════════════════════════════════
import time
print('✅ XIOSYNC worker is running. You may close the browser.')
while True:
    time.sleep(60)